# cross-product-normal — worked example 3: Per-vertex normals by scattering face normals to a mesh

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-product-normal`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Smooth shading needs a normal per *vertex*, not per *face*. The standard trick: compute each face's (un-normalized) cross-product normal so its magnitude weights by area, scatter-add each face normal onto the three vertices it touches, then normalize each accumulated vertex normal. Area weighting makes large faces influence shared vertices more.

## Worked solution

**Step 1 — gather face vertex positions.** `faces` is `(F, 3)` indices into `verts` `(V, 3)`. Indexing `verts[faces]` gives `(F, 3, 3)`: for each face, its three 3-D vertex positions.

**Step 2 — batched edges + cross.** `e1 = tri[:,1] - tri[:,0]`, `e2 = tri[:,2] - tri[:,0]`, then `face_n = cross(e1, e2, dim=-1)` is `(F, 3)`. We deliberately do **not** normalize here: the magnitude equals 2×area, which is exactly the area weight we want.

**Step 3 — scatter-add to vertices.** Each face touches its three vertices, so we add the same `face_n` to all three rows of an accumulator `vert_n` of shape `(V, 3)`. `index_add_` with the flattened face indices and `face_n` repeated 3 times does this in one call.

**Step 4 — normalize per vertex.** After accumulation each vertex holds the area-weighted sum of its incident face normals. Dividing by the per-row norm (clamped to avoid divide-by-zero on isolated vertices) yields unit vertex normals `(V, 3)`.

In [ ]:
def vertex_normals(verts: Tensor, faces: Tensor) -> Tensor:
    tri = verts[faces]                              # (F, 3, 3)
    e1 = tri[:, 1] - tri[:, 0]                      # (F, 3)
    e2 = tri[:, 2] - tri[:, 0]                      # (F, 3)
    face_n = t.linalg.cross(e1, e2, dim=-1)         # (F, 3) area-weighted
    vert_n = t.zeros_like(verts)                    # (V, 3)
    idx = faces.reshape(-1)                         # (3F,)
    contrib = repeat(face_n, 'f c -> (f three) c', three=3)
    vert_n.index_add_(0, idx, contrib)
    norms = vert_n.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    return vert_n / norms

verts = t.tensor([[0.0, 0.0, 0.0],
                  [1.0, 0.0, 0.0],
                  [0.0, 1.0, 0.0],
                  [1.0, 1.0, 0.0]])
faces = t.tensor([[0, 1, 2], [1, 3, 2]])
print(vertex_normals(verts, faces))